# Day 08：面向对象编程（上）（Jupyter Notebook 实战手册）

> **所属阶段**：第 1 阶段 · 第一阶段：Python 编程基础 ｜ **第 2 周**：第 2 周：面向对象、工程化与命令行 AI 助手实战  
> **GitLab 工单**：`STARMIND-008` ｜ **开发分支**：`feat/day08-oop-chat-message`  
> **配套 HTML5 课件**：`courseware/day08/index.html` ｜ **独立版本快照**：`versions/day08/`

---

## 🎙️ 老陈开场旁白

新的一周开始了！上周咱们用裸字典 `{'role': 'user', 'content': '你好'}` 表示消息，写着写着你肯定发现痛点了：万一有人把 `'role'` 拼成 `'roles'`，或者塞了个非法角色 `'boss'` 进去，裸字典根本拦不住！今天 STARMIND-008 咱们用面向对象（OOP）彻底治好这个毛病：设计星辰智脑最核心的数据类 `ChatMessage`，把字段校验、时间戳、Token 估算、转 API 字典（`to_api_dict`）、工厂类方法（`@classmethod`）和静态校验工具（`@staticmethod`）全优雅地封装进去！

### 承上启下
- **昨日基线**：第一周咱们用字典和函数交付了企业通讯录系统。但随着业务变复杂，满天飞的裸字典没有强约束，少写一个字段直到运行才炸。
- **明日预告**：明天（Day 9）咱们继续深挖 OOP 进阶：继承、多态、super、魔术方法与 `@property`，亲手搭出 `BaseModel -> OpenAIModel / QwenModel` 多模型统一适配层！


In [ ]:
# 环境自检与路径初始化：确保可以无缝导入当日版本模块并支持异步协程运行
import os
import sys
from pathlib import Path

# 允许在 Jupyter 已有事件循环内安全调用 asyncio.run()
try:
    import nest_asyncio
    nest_asyncio.apply()
except ImportError:
    pass

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'versions').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DAY_DIR = PROJECT_ROOT / 'versions' / 'day08'
for _p in (DAY_DIR, DAY_DIR / 'src', PROJECT_ROOT):
    if _p.exists() and str(_p) not in sys.path:
        sys.path.insert(0, str(_p))
os.chdir(DAY_DIR)
__file__ = str(DAY_DIR / 'src' / 'notebook_entry.py')

print(f'✅ Python 版本: {sys.version.split()[0]}')
print(f'✅ 项目根目录: {PROJECT_ROOT}')
print(f'✅ 当日版本目录: {DAY_DIR}')


## 📋 企业需求背景与验收目标（STARMIND-008）

星辰智脑对话引擎需要统一的消息领域对象，替代容易拼错键名的裸字典，并支持单会话多轮记忆管理、Token 统计与上下文窗口裁剪。

**今日迭代目标：**
- 理解类（Class）与实例（Instance）、`__init__` 构造方法与 `self` 绑定机制。
- 掌握实例方法、类方法（`@classmethod`）与静态方法（`@staticmethod`）的区别与应用场景。
- 完成 `ChatMessage` 与 `ChatSession` 两个核心领域类并通过单元测试。

**验收标准：**
- `ChatMessage` 在初始化时校验合法角色与非空内容，提供 `to_api_dict()` 与 `from_dict()` 工厂方法。
- `ChatSession` 支持追加消息、保留 system prompt 的窗口裁剪、总 Token 统计及导出 OpenAI messages 列表。
- 所有正常与异常用例通过 pytest。

---

## 📚 课堂核心技术笔记

### 1. 1. 类、实例与 self 的本质

类是蓝图，实例是根据蓝图造出来的具体对象。`__init__(self, ...)` 在对象创建后自动执行初始化。`self` 就是当前实例本身的引用，调用 `msg.to_api_dict()` 时 Python 会自动把 `msg` 作为第一个参数 `self` 传进去。

> **⚠️ 老陈避坑**：千万不要把可变容器（如 `messages = []`）直接写在类体顶层当作实例属性！那样所有会话实例会共享同一个列表，导致不同用户聊天记录互相串台！实例独享的可变容器必须写在 `__init__` 的 `self.messages = []` 里。

### 2. 2. 实例方法 vs @classmethod vs @staticmethod

- **实例方法** `(self, ...)`：读写当前对象的状态（如 `msg.to_api_dict()`）。
- **类方法** `@classmethod (cls, ...)`：接收类本身 `cls`，最常用来写「替代构造器/工厂方法」（如 `ChatMessage.from_dict(data)`），且子类继承时会自动返回子类实例。
- **静态方法** `@staticmethod (...)`：既不需要 `self` 也不需要 `cls`，纯粹归类到类命名空间下的工具函数（如 `ChatMessage.estimate_tokens(text)`）。

> **⚠️ 老陈避坑**：在 `@classmethod` 中构造对象请写 `cls(...)` 而不是写死类名 `ChatMessage(...)`，这样以后被子类继承才不会返回错误类型。

### 3. 3. 封装与不变式（Invariant）保护

把数据合法性检查放在 `__init__` 入口，就能保证内存中存在的每一个 `ChatMessage` 对象都是 100% 合法的，下游代码无需再反复判空或查角色拼写。

> **⚠️ 老陈避坑**：约定俗成：以单下划线开头的属性（如 `self._messages`）表示内部受保护状态，外部应通过公开方法操作而非直接乱改。


## 💻 实操步骤 1：步骤一：实现星辰智脑核心领域类 ChatMessage

- **目标文件**：`versions/day08/src/day08_chat_message.py`
- **老陈代码旁白**：咱们亲手写星辰智脑的第一个正式领域类 `ChatMessage`，把角色校验、Token 估算、JSON 工厂构造全部优雅封装。


In [ ]:
"""
STARMIND-008 模块一：核心领域模型 ChatMessage。
演示类属性、__init__ 初始化与不变式校验、实例方法、@classmethod 与 @staticmethod。
"""

from datetime import datetime, timezone
from typing import Dict, Any, Tuple


class ChatMessage:
    """表示大模型多轮对话中的单条标准化消息。"""

    # 类属性：所有消息实例共享的合法角色白名单（不可变元组）
    VALID_ROLES: Tuple[str, ...] = ("system", "user", "assistant", "tool")

    def __init__(self, role: str, content: str, created_at: str | None = None) -> None:
        normalized_role = role.strip().lower()
        # 构造期不变式校验：杜绝非法角色进入系统
        if normalized_role not in self.VALID_ROLES:
            raise ValueError(f"非法角色 '{role}'，允许的角色: {self.VALID_ROLES}")

        cleaned_content = content.strip()
        if not cleaned_content:
            raise ValueError("消息正文 content 不能为空")

        # 实例属性：每条消息独立拥有
        self.role: str = normalized_role
        self.content: str = cleaned_content
        self.created_at: str = created_at or datetime.now(timezone.utc).isoformat(timespec="seconds")
        self.tokens: int = self.estimate_tokens(self.content)

    @staticmethod
    def estimate_tokens(text: str) -> int:
        """静态工具方法：粗略估算文本 Token 数量。"""
        if not text:
            return 0
        return max(1, (len(text) * 3 + 3) // 4)

    def to_api_dict(self) -> Dict[str, str]:
        """实例方法：转换为 OpenAI/DeepSeek 兼容的最小 API 消息字典。"""
        return {"role": self.role, "content": self.content}

    def to_storage_dict(self) -> Dict[str, Any]:
        """实例方法：转换为包含元数据的完整持久化字典。"""
        return {
            "role": self.role,
            "content": self.content,
            "created_at": self.created_at,
            "tokens": self.tokens,
        }

    @classmethod
    def from_dict(cls, data: Dict[str, Any]) -> "ChatMessage":
        """类方法工厂：从字典反序列化构造 ChatMessage 实例。"""
        return cls(
            role=str(data.get("role", "")),
            content=str(data.get("content", "")),
            created_at=data.get("created_at"),
        )

    def __repr__(self) -> str:
        preview = self.content if len(self.content) <= 20 else self.content[:17] + "..."
        return f"ChatMessage(role={self.role!r}, tokens={self.tokens}, content={preview!r})"


if __name__ == "__main__":
    m1 = ChatMessage("user", "你好，请介绍一下杭州星辰数科")
    m2 = ChatMessage.from_dict({"role": "assistant", "content": "您好！星辰数科专注于企业级大模型应用研发。"})
    print(m1)
    print("API 协议格式:", m1.to_api_dict())
    print("反序列化对象:", m2)


### 🔍 步骤 1 代码深度解析

1. `VALID_ROLES` 用元组定义在类上，全员共享且不可篡改。
2. `from_dict` 使用 `cls(...)` 而不是 `ChatMessage(...)`，保证明天做继承时子类也能直接复用这个工厂方法。
3. `__repr__` 截断超长正文并展示角色和 Token 数，调试时打印列表非常清爽。

**预期运行输出参考：**
```text
ChatMessage(role='user', tokens=11, content='你好，请介绍一下杭州星辰数科')
API 协议格式: {'role': 'user', 'content': '你好，请介绍一下杭州星辰数科'}
反序列化对象: ChatMessage(role='assistant', tokens=17, content='您好！星辰数科专注于企业级大模型...')
```


## 💻 实操步骤 2：步骤二：实现多轮会话记忆容器 ChatSession

- **目标文件**：`versions/day08/src/day08_chat_session.py`
- **老陈代码旁白**：有了单条消息 `ChatMessage`，咱们再用「组合（Composition）」封装一个 `ChatSession` 会话管理器，自动管理历史记忆和窗口裁剪。


In [ ]:
"""
STARMIND-008 模块二：多轮会话记忆容器 ChatSession。
演示对象组合（ChatSession 持有多个 ChatMessage）、窗口裁剪与受保护状态管理。
"""

from typing import List, Dict, Optional
from day08_chat_message import ChatMessage


class ChatSession:
    """管理单个用户的多轮对话上下文与滑动窗口裁剪。"""

    def __init__(
        self,
        session_id: str,
        system_prompt: Optional[str] = None,
        max_history: int = 10,
    ) -> None:
        self.session_id: str = session_id
        self.max_history: int = max(1, int(max_history))
        # 系统指令单独保管，裁剪历史时绝不误删
        self._system_msg: Optional[ChatMessage] = (
            ChatMessage("system", system_prompt) if system_prompt else None
        )
        # 实例独享的消息历史列表（绝不能写成类属性！）
        self._messages: List[ChatMessage] = []

    def add_message(self, role: str, content: str) -> ChatMessage:
        """向会话追加一条非 system 消息，并在超出窗口上限时自动裁剪最早消息。"""
        msg = ChatMessage(role, content)
        if msg.role == "system":
            self._system_msg = msg
            return msg

        self._messages.append(msg)
        # 滑动窗口裁剪：仅保留最近 max_history 条对话
        if len(self._messages) > self.max_history:
            self._messages = self._messages[-self.max_history :]
        return msg

    def total_tokens(self) -> int:
        """统计当前会话上下文（含系统指令）的估算总 Token 数。"""
        sys_tokens = self._system_msg.tokens if self._system_msg else 0
        return sys_tokens + sum(m.tokens for m in self._messages)

    def to_api_messages(self) -> List[Dict[str, str]]:
        """导出可直接传给大模型 API 的 messages 字典列表。"""
        payload: List[Dict[str, str]] = []
        if self._system_msg:
            payload.append(self._system_msg.to_api_dict())
        payload.extend(m.to_api_dict() for m in self._messages)
        return payload

    def clear(self) -> None:
        """清空历史对话，但保留 system_prompt 人设设定。"""
        self._messages.clear()


if __name__ == "__main__":
    session = ChatSession("sess-001", system_prompt="你是星辰智脑助手", max_history=2)
    session.add_message("user", "第一句提问")
    session.add_message("assistant", "第一句回复")
    session.add_message("user", "第二句提问")
    print("当前窗口消息数(不含system):", len(session.to_api_messages()) - 1)
    print("总 Token 估算:", session.total_tokens())
    print("导出 API Payload:", session.to_api_messages())


### 🔍 步骤 2 代码深度解析

1. 把 `_system_msg` 和 `_messages` 分开存储是极其聪明的架构决定：无论对话进行了 100 轮还是用户执行了 `/clear`，AI 的系统人设都不会被冲掉。
2. `self._messages[-self.max_history:]` 用负向切片一行完成滑动窗口截断。

**预期运行输出参考：**
```text
当前窗口消息数(不含system): 2
总 Token 估算: 16
导出 API Payload: [{'role': 'system', 'content': '你是星辰智脑助手'}, {'role': 'assistant', 'content': '第一句回复'}, {'role': 'user', 'content': '第二句提问'}]
```


## 🧪 当日自动化单元测试

在终端运行 `pytest versions/day08/tests/test_day08.py -v` 即可验收全部测试断言。下面展示完整测试脚本源码，供对照学习测试写法：

```python
"""STARMIND-008 自动化单元测试。"""

import sys
from pathlib import Path
import pytest

ROOT = Path(__file__).resolve().parent.parent
for p in (ROOT, ROOT / "src"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from day08_chat_message import ChatMessage
from day08_chat_session import ChatSession


def test_chat_message_validation_and_factory():
    msg = ChatMessage("USER", "  测试消息  ")
    assert msg.role == "user"
    assert msg.content == "测试消息"
    assert msg.to_api_dict() == {"role": "user", "content": "测试消息"}

    rebuilt = ChatMessage.from_dict(msg.to_storage_dict())
    assert rebuilt.role == "user"
    assert rebuilt.content == "测试消息"

    with pytest.raises(ValueError):
        ChatMessage("boss", "非法角色")
    with pytest.raises(ValueError):
        ChatMessage("user", "   ")


def test_chat_session_window_and_isolation():
    s1 = ChatSession("s1", system_prompt="人设A", max_history=2)
    s2 = ChatSession("s2", system_prompt="人设B", max_history=2)

    s1.add_message("user", "q1")
    s1.add_message("assistant", "a1")
    s1.add_message("user", "q2")

    # s1 窗口裁剪后应保留 system + a1 + q2
    api_msgs = s1.to_api_messages()
    assert [m["content"] for m in api_msgs] == ["人设A", "a1", "q2"]
    # s2 应完全隔离不受 s1 影响
    assert [m["content"] for m in s2.to_api_messages()] == ["人设B"]

    s1.clear()
    assert [m["content"] for m in s1.to_api_messages()] == ["人设A"]

```


## 📝 课后梯度实战作业与标准答案

建议先在下方新建空白单元格独立完成编码，运行通过后再核对参考答案。


### 作业 1（基础巩固）：基础巩固：API Token 账单计数器类

**题目要求：**
设计类 `TokenTracker`，记录每次调用的输入与输出 Token 数，提供 `record(in_tokens, out_tokens)`、`total_tokens()` 和 `estimated_rmb(in_price=0.002, out_price=0.008)` 方法。

**思路提示：** 在 `__init__` 中初始化 `self.input_total = 0` 和 `self.output_total = 0`。


In [ ]:
"""Day 8 课后题 1：Token 账单追踪器类。"""


class TokenTracker:
    """累计单个会话或用户的 Token 消耗并估算人民币费用。"""

    def __init__(self) -> None:
        self.input_total: int = 0
        self.output_total: int = 0
        self.call_count: int = 0

    def record(self, in_tokens: int, out_tokens: int) -> None:
        """记录单次调用消耗。"""
        self.input_total += max(0, int(in_tokens))
        self.output_total += max(0, int(out_tokens))
        self.call_count += 1

    def total_tokens(self) -> int:
        return self.input_total + self.output_total

    def estimated_rmb(self, in_price_per_1k: float = 0.002, out_price_per_1k: float = 0.008) -> float:
        cost = (self.input_total / 1000.0) * in_price_per_1k + (self.output_total / 1000.0) * out_price_per_1k
        return round(cost, 6)


if __name__ == "__main__":
    tracker = TokenTracker()
    tracker.record(500, 200)
    tracker.record(800, 400)
    print(f"调用 {tracker.call_count} 次，总 Token: {tracker.total_tokens()}，费用: ¥{tracker.estimated_rmb()}")


**答案解析与评分标准：**

典型有状态计数器类，将状态与计费逻辑内聚在一起，后续可直接挂载到 `ChatSession` 中。


### 作业 2（企业实战）：企业实战：带过期标记的提示词模板类

**题目要求：**
设计 `PromptAsset` 类，包含 `name`、`version`、`template` 和类计数器 `total_created`（记录一共创建过多少个模板实例），提供 `render(**kwargs)` 方法与 `@classmethod get_total_created()`。

**思路提示：** 在 `__init__` 中执行 `type(self).total_created += 1`。


In [ ]:
"""Day 8 课后题 2：带实例计数的提示词资产类。"""


class PromptAsset:
    """企业级 Prompt 资产管理对象。"""

    total_created: int = 0

    def __init__(self, name: str, version: str, template: str) -> None:
        self.name = name
        self.version = version
        self.template = template
        type(self).total_created += 1

    def render(self, **kwargs) -> str:
        return self.template.format(**kwargs)

    @classmethod
    def get_total_created(cls) -> int:
        return cls.total_created


if __name__ == "__main__":
    p1 = PromptAsset("客服问候", "v1.0", "你好 {user}，请问有什么可以帮您？")
    p2 = PromptAsset("工单总结", "v1.2", "请用一句话总结：{text}")
    print(p1.render(user="林产品"))
    print("累计创建模板数:", PromptAsset.get_total_created())


**答案解析与评分标准：**

区分了类级共享状态 `total_created` 与实例独享状态 `name/version/template`。


### 作业 3（架构进阶）：架构进阶：支持上下文 Token 预算自动裁剪的会话类

**题目要求：**
比按条数裁剪更贴近生产的是「按总 Token 上限裁剪」。设计 `BudgetSession`，当新增消息后总 Token 超过 `max_tokens` 时，持续弹出最早的消息直到满足预算（至少保留最新一条用户消息）。

**思路提示：** 用 `while len(self._messages) > 1 and self.total_tokens() > self.max_tokens: self._messages.pop(0)`。


In [ ]:
"""Day 8 课后题 3：按 Token 预算动态裁剪的会话容器。"""

from typing import List


class BudgetSession:
    """按最大 Token 预算自动淘汰最早历史消息的会话管理器。"""

    def __init__(self, max_tokens: int = 30) -> None:
        self.max_tokens = max_tokens
        self._messages: List[str] = []

    @staticmethod
    def _count(text: str) -> int:
        return max(1, len(text))

    def total_tokens(self) -> int:
        return sum(self._count(m) for m in self._messages)

    def add(self, text: str) -> None:
        self._messages.append(text)
        # 持续淘汰最早消息，但至少保留刚输入的最新一条
        while len(self._messages) > 1 and self.total_tokens() > self.max_tokens:
            self._messages.pop(0)

    def messages(self) -> List[str]:
        return list(self._messages)


if __name__ == "__main__":
    bs = BudgetSession(max_tokens=15)
    for m in ["消息一12345", "消息二12345", "消息三12345"]:
        bs.add(m)
    print("裁剪后保留:", bs.messages(), "当前总Token:", bs.total_tokens())


**答案解析与评分标准：**

真实大模型上下文窗口（如 8K/32K/128K）是以 Token 计费和截断的，本题用 `while` 循环实现了生产级预算裁剪。


---

## 🎯 老陈收尾复盘

今天你亲手写出了 `ChatMessage` 和 `ChatSession`，这俩类咱们后面从 Day 9 一直用到 Day 14 甚至 FastAPI 后端！记住三种方法的分工：操作单个对象状态用实例方法（`self`），做构造工厂或读类属性用 `@classmethod`（`cls`），纯工具函数用 `@staticmethod`。明天咱们玩继承和多态，把不同大模型厂商统一收编！
